# Insurance Fraud Analytics & Claims Portfolio Monitoring

**Source:** Minitab Insurance Fraud dataset

This notebook demonstrates data cleaning, exploratory analysis, statistical diagnostics and baseline fraud-risk modelling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df=pd.read_csv('insurance_fraud_clean.csv')
df.head()

## 1. Data quality

The source uses `*` as a missing-value marker. The cleaned dataset converts these markers to missing values and validates key numeric fields.

In [ ]:
print('Rows:',len(df))
print('Columns:',len(df.columns))
print('\nMissing values:')
print(df.isna().sum().sort_values(ascending=False).head(15))

## 2. Portfolio KPIs

In [ ]:
valid=df[df['fraud_flag'].notna()].copy()
print('Claims with known fraud outcome:',len(valid))
print('Total claim value: ${:,.0f}'.format(valid.total_claim.sum()))
print('Average claim: ${:,.0f}'.format(valid.total_claim.mean()))
print('Fraud rate: {:.2f}%'.format(valid.fraud_flag.mean()*100))

## 3. Business segmentation

In [ ]:
channel=valid.groupby('channel').agg(claims=('claim_number','count'),avg_claim=('total_claim','mean'),avg_days_open=('days open','mean'),fraud_rate=('fraud_flag','mean')).sort_values('claims',ascending=False)
channel['fraud_rate_pct']=channel['fraud_rate']*100
channel

In [ ]:
site=valid.groupby('accident_site').agg(claims=('claim_number','count'),total_claim_value=('total_claim','sum'),avg_claim=('total_claim','mean'),fraud_rate=('fraud_flag','mean')).sort_values('total_claim_value',ascending=False)
site['fraud_rate_pct']=site['fraud_rate']*100
site

In [ ]:
monthly=valid.groupby('claim_month').agg(claims=('claim_number','count'),total_claim_value=('total_claim','sum'),avg_claim=('total_claim','mean')).reset_index()
monthly.head()

## 4. Predictive modelling

A baseline logistic regression model estimates fraud probability. Categorical variables are one-hot encoded and numeric variables are imputed/scaled. Class weighting addresses imbalance.

In [ ]:
features=['age_of_driver', 'gender', 'marital_status', 'safety_rating', 'annual_income', 'high_education', 'address_change', 'property_status', 'claim_day_of_week', 'accident_site', 'past_num_of_claims', 'witness_present', 'liab_prct', 'channel', 'police_report', 'age_of_vehicle', 'vehicle_category', 'vehicle_price', 'vehicle_color', 'total_claim', 'injury_claim', 'policy deductible', 'annual premium', 'days open', 'form defects']
X=valid[features]
y=valid['fraud_flag'].astype(int)
cat=[c for c in features if X[c].dtype=='object']
num=[c for c in features if c not in cat]
pre=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),num),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
model=Pipeline([('preprocess',pre),('classifier',LogisticRegression(max_iter=1500,class_weight='balanced'))])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
model.fit(Xtr,ytr)
prob=model.predict_proba(Xte)[:,1]
print('ROC-AUC:',round(roc_auc_score(yte,prob),3))

## 5. Management interpretation

The project treats the analysis as a portfolio-monitoring workflow: claim exposure, operational performance, fraud indicators and predictive risk are evaluated together. Statistical relationships are treated as evidence for further investigation rather than causal proof.